In [ ]:
# =============================================================================
# Section 3 Results Figures
#
# PURPOSE:
# This notebook generates figures for the results section using participant-level
# speech, pause, and standardized score data.
#
# INPUTS:
# - CWNS_CWS_all_details.csv
# - p-value outputs from the group statistics notebooks
#
# OUTPUTS:
# - Boxplots comparing CWS and CWNS groups
# - Correlation plots
# - Combined results figures
#
# ANALYSIS OVERVIEW:
# 1. Load participant-level analysis dataset
# 2. Prepare group comparison and p-value annotation helpers
# 3. Generate boxplots for selected speech/pause metrics
# 4. Generate correlation plots for significant or relevant associations
# 5. Generate combined figures for manuscript/results reporting
#
# NOTE:
# This notebook should be run after the statistical analysis notebooks so that
# p-values are available for figure annotation.
# =============================================================================

In [ ]:
import os
import numpy as np, pandas as pd, matplotlib.pyplot as plt

from decimal import Decimal, getcontext, ROUND_HALF_UP
from itertools import combinations
from scipy.stats import pearsonr
from sklearn.linear_model import LinearRegression

In [ ]:
# Load participant-level dataset containing demographics, speech/pause metrics,
# fluency scores, and long/short pause measures

# Project directory structure
#
# ATAS_CWNS_CWS_Project/        ← THIS is base_dir
# │
# ├── Audio_files_clipped/      ← create this folder; clipped audio files stored here
# ├── Visualize/                ← create this folder; .wav audio files for visualization/analysis stored here
# ├── Stat_csv_files/           ← input CSV files and summary statistics CSV files stored here
# ├── Individual_OutputCSV_Files/ ← individual output CSV files stored here
# │
# ├── Notebooks/                ← notebooks stored here
# │   ├── ATAS_Visualization/   ← visualization notebook
# │   └── ML_Analysis/          ← machine learning analysis notebooks
# │
# └── Scripts/                  ← function notebooks
#
base_dir = '..../ATAS_CWNS_CWS_Project'  # Change to required project directory


# Load the csv file with all participant details 
#(Includes - participant details, SSI_score details,Speech_rate, and long and short pause metrics)
par_details = pd.read_csv(os.path.join(base_dir, "Stat_csv_files","CWNS_CWS_all_details.csv")) #Change path as required

# Load the p-value results generated in the Group_Stats_and_GLM_models.ipynb
pval_file = base_dir + '/Stat_csv_files/pval.csv'  # Change path as required
pval_table_org = pd.read_csv(pval_file)

In [ ]:
def custom_round(x):
    try:
        if pd.isna(x):
            return x
        x = float(x)
    except Exception:
        return x

    # identical logic, compact:
    return int(x) + ((x - int(x)) > 0.5)


# Apply to Age column (same logic/results)
par_details['Age'] = par_details['Age'].apply(custom_round)

df = par_details.copy()
group_col = 'Group'

xdata = df  # Assuming data.csv exists in the specified directory

# Compare means (grouped by condition)
xagg = xdata.groupby("Group").mean(numeric_only=True).reset_index()
# print(xagg)  # Display aggregated data


# Create a copy of the original table 
pval_table = pval_table_org.copy()

# Remove 'log_' prefix if it exists (robust to non-string values)
pval_table['Metric'] = pval_table['Metric'].astype(str).str.replace('^log_', '', regex=True)

# This is what is later used to build df_pval
data_pval = pval_table[['Metric', 'p-value']].copy()

# ---- Optional display table with cascading rounding logic (does NOT change pval_table used later) ----
def format_p_value(p):
    try:
        if pd.isna(p):
            return float('nan')
        p = float(p)
    except Exception:
        return float('nan')

    getcontext().prec = 20
    d = Decimal(str(p))
    high_precision = d.quantize(Decimal('0.0000000001'), rounding=ROUND_HALF_UP)
    final = high_precision.quantize(Decimal('0.01'), rounding=ROUND_HALF_UP)

    if final >= Decimal('0.01'):
        return float(final)
    elif final == Decimal('0.00') and d != Decimal('0.00'):
        return float(d.normalize())
    else:
        return float(final)


df_pval = data_pval.copy()
df_pval["Rounded_p"] = df_pval["p-value"].apply(format_p_value)
df_pval["Rounded_p"] = df_pval["Rounded_p"].apply(
    lambda x: round(x, 3) if (pd.notna(x) and x < 0.01) else (round(x, 2) if pd.notna(x) else x)
)

print(df_pval)


In [ ]:
def get_group_comparisons_table(pval_table):
    """
    Returns a tidy table of ONLY the Group (CWS vs CWNS) comparisons + their p-values.
    Works whether Predictor uses 'Group[T.CWS]' or 'Group_CWS' or Predictor is missing.
    """
    ptab = pval_table.copy()

    if 'Predictor' in ptab.columns:
        group_mask = ptab['Predictor'].isin(['Group[T.CWS]', 'Group_CWS'])
        out = ptab.loc[group_mask, ['Metric', 'Predictor', 'p-value']].copy()
    else:
        out = ptab.loc[:, ['Metric', 'p-value']].copy()
        out['Predictor'] = 'Group'

    # ensure numeric p-values where possible
    out['p-value'] = pd.to_numeric(out['p-value'], errors='coerce')

    out = out.sort_values('p-value', na_position='last').reset_index(drop=True)
    out['significant_(p<0.05)'] = out['p-value'] < 0.05
    return out

# --- Run this ---
group_pvals_all = get_group_comparisons_table(pval_table)
display(group_pvals_all)

group_pvals_sig = group_pvals_all[group_pvals_all['p-value'] < 0.05].reset_index(drop=True)
display(group_pvals_sig)


## Define all boxplots to generate (Figure 2)
## Each entry specifies the data column, display label, unit conversion, and title

In [ ]:
# List every boxplot you want to potentially generate
BOXPLOT_SPECS = [
    dict(column_name='Mean Pause_s',           metric_name='Mean_Pause_s_ms',           y_ax_label='Time (ms)',  unit_conversion_factor=1000, plot_title_input='Mean Pause Duration'),
    dict(column_name='Mean Speech_s',          metric_name='Mean_Speech_s_ms',          y_ax_label='Time (ms)',  unit_conversion_factor=1000, plot_title_input='Mean Vocal Duration'),
    dict(column_name='Speech_Rate',            metric_name='Speech_Rate',               y_ax_label='Words/min',  unit_conversion_factor=1,    plot_title_input=' Speech Rate'),
    dict(column_name='Pause_Duration_s',       metric_name='Pause_Duration_s',          y_ax_label='Time (s)',   unit_conversion_factor=1,    plot_title_input="Total Pause Time"),
    dict(column_name='CV Speech',              metric_name='CV_Speech',                 y_ax_label='Coefficient of Variation', unit_conversion_factor=1, plot_title_input='Vocal Duration Variability'),
    dict(column_name='Pause_Events',           metric_name='Pause_Events',              y_ax_label='No. of Pauses', unit_conversion_factor=1,   plot_title_input='Pause Count'),
    dict(column_name='long_p_count',           metric_name='long_p_count',              y_ax_label='No. of Pauses', unit_conversion_factor=1,   plot_title_input="Long Pause Count"),
    dict(column_name='short_p_count',          metric_name='short_p_count',             y_ax_label='No. of Pauses', unit_conversion_factor=1,   plot_title_input="Short Pause Count"),
    dict(column_name='short_p_durations_mean', metric_name='short_p_durations_mean_ms', y_ax_label='Time (ms)',  unit_conversion_factor=1000, plot_title_input="Mean Short Pause Duration"),
    dict(column_name='short_p_durations_cv',   metric_name='short_p_durations_cv',      y_ax_label='Time (ms)',  unit_conversion_factor=1000, plot_title_input="Short Pause Duration Variability"),
]
def plot_metric_from_pval(column_name, metric_name, y_ax_label, df, group_col, pval_table, plot_figure=1, unit_conversion_factor=1, plot_title_input=None):
    """
    Plot CWS vs. CWNS data using a title from the corresponding row in a p-value table and the p-value from the 'p' column.
    
    Parameters:
    df (DataFrame): The main DataFrame containing participant data.
    column_name (str): The metric column to be compared.
    group_col (str): The column indicating the group ('CWS' or 'CWNS').
    pval_table (DataFrame): Table containing p-values and corresponding metric names.
    metric_name (str): (The metric name to be used as the plot title - check) - this is from the pval column.
    plot_figure (int, optional): 1 to plot the figure, 0 to skip plotting. Default is 1 (plot).
    unit_conversion_factor (float, optional): Factor to convert units. Default is 1 (no conversion).
    plot_title_input (str, optional): Optional title input by the user. If None, the title from the pval_table will be used.
    
    Returns:
    None
    """
    # Extract CWS and CWNS data
    AWS_data_chk = df[df[group_col] == 'CWS'][column_name] * unit_conversion_factor
    AWNS_data_chk = df[df[group_col] == 'CWNS'][column_name] * unit_conversion_factor
    
    # Retrieve the corresponding metric name for the plot title and p-value for 'Group[T.CWS]'
    # Check if the 'Predictor' column exists in pval_table
    if 'Predictor' in pval_table.columns:
        selected_row = pval_table[(pval_table['Metric'] == metric_name) & ((pval_table['Predictor'] == 'Group[T.CWS]') | (pval_table['Predictor'] == 'Group_CWS'))]

    else:
        selected_row = pval_table[pval_table['Metric'] == metric_name]   
    plot_title = plot_title_input if plot_title_input else selected_row['Metric'].values[0] if not selected_row.empty else metric_name
    p_value = selected_row['p-value'].values[0] if not selected_row.empty else 'N/A'
    
    if plot_figure == 1:
        data = [AWNS_data_chk, AWS_data_chk]
        fig = plt.figure(figsize=(5, 4))
        ax = fig.add_axes([0, 0, 1, 1])
        bp = ax.boxplot(data, patch_artist=True)
        
        # Set colors for the box plots
        colors = ['yellowgreen', 'cornflowerblue']
        for patch, color in zip(bp['boxes'], colors):
            patch.set_facecolor(color)
        for median in bp['medians']:
            median.set(color='black', linewidth=3)
        
        # Customize the plot
        plt.title(f"{plot_title}", fontsize=20)
        plt.xticks([1, 2], ['CWNS', 'CWS'], fontsize=20)
        plt.yticks(fontsize=15)
        plt.ylabel(y_ax_label, fontsize=20)
        # Significance level and annotation
        bottom, top = ax.get_ylim()
        y_range = top - bottom
        bar_height = (y_range * 0.07 * -1) + top


        # 1. Keep raw p-value as float for logic
        raw_p_value = p_value

        # 2. Format for display
        #formatted_p = float(format_p_value(raw_p_value))

        # 3. Significance stars based on raw value
        if p_value < 0.001:
            sig_symbol = '***'
        elif p_value < 0.01:
            sig_symbol = '**'
        elif p_value < 0.05:
            sig_symbol = '*'
        else:
            sig_symbol = ''

        # 4. Output
        print(f"p_value: {p_value}")  # e.g., p_value: 0.01

        # 5. Plot
        text_height = bar_height - (y_range * 0.1)
        plt.text(1.5, text_height, sig_symbol, ha='center', va='bottom', color='k', fontsize=25)
        plt.show()


def plot_only_significant_boxplots(df, group_col, pval_table, specs, alpha=0.05):
    """
    Uses plot_metric_from_pval() to plot ONLY those metrics whose Group p-value < alpha.
    Also prints + returns the p-value summary table for the plotted metrics.
    """
    # group comparison p-values
    group_pvals_all = get_group_comparisons_table(pval_table)

    # pvals only for the metrics we intend to consider (from specs)
    spec_metrics = pd.Series([s['metric_name'] for s in specs], name='Metric').unique().tolist()
    pvals_for_specs = group_pvals_all[group_pvals_all['Metric'].isin(spec_metrics)].copy()
    pvals_for_specs = pvals_for_specs.sort_values('p-value', na_position='last').reset_index(drop=True)

    print("P-values for all boxplot comparisons (your spec list):")
    display(pvals_for_specs)

    sig_metrics = set(pvals_for_specs.loc[pvals_for_specs['p-value'] < alpha, 'Metric'].tolist())

    print(f"Significant metrics (p < {alpha}):")
    display(pvals_for_specs[pvals_for_specs['p-value'] < alpha].reset_index(drop=True))

    # Plot only significant
    for s in specs:
        do_plot = 1 if s['metric_name'] in sig_metrics else 0
        if do_plot == 1:
            plot_metric_from_pval(
                s['column_name'],
                s['metric_name'],
                s['y_ax_label'],
                df,
                group_col,
                pval_table,
                plot_figure=1,
                unit_conversion_factor=s.get('unit_conversion_factor', 1),
                plot_title_input=s.get('plot_title_input', None),
            )

    return pvals_for_specs

# --- Run this ---
pvals_used_for_boxplots = plot_only_significant_boxplots(df, group_col, pval_table, BOXPLOT_SPECS, alpha=0.05)


In [ ]:

def analyze_and_plot_correlation(x, y, x_label="X-axis", y_label="Y-axis", lp1=0.05, lp2=0.95):

    # Calculate the Pearson correlation coefficient
    corr_coeff, p_value = pearsonr(x, y)
    print("Correlation coefficient:", corr_coeff)
    print("p-value:", p_value)
    
    # Fit a linear regression model
    x1 = np.array(x).reshape(-1, 1)  # Reshape for scikit-learn compatibility
    model = LinearRegression().fit(x1, y)

    # Create a scatter plot of the data
    plt.figure(figsize=(10, 5))
    plt.scatter(x, y, alpha=0.7)

    # Add a regression line
    plt.plot(np.unique(x), model.predict(np.unique(x).reshape(-1, 1)), color='black')

    # Add axis labels
    plt.xlabel(x_label, fontsize=20)
    plt.ylabel(y_label, fontsize=20)
    plt.xticks(fontsize=15)
    plt.yticks(fontsize=15)

    # Add correlation coefficient and p-value inside the plot

    plt.text(lp1, lp2, f"r: {corr_coeff:.2f}, p: {'< .01' if p_value < 0.01 else f'{p_value:.2f}'.lstrip('0')}", 
             fontsize=15, ha='left', va='top', fontweight='bold', style='italic', transform=plt.gca().transAxes)


    plt.show()

    return {
        "correlation_coefficient": corr_coeff,
        "p_value": p_value,
        "regression_model": model
    }


def analyze_and_plot_correlation_unq(x, y, x_label="X-axis", y_label="Y-axis", lp1=0.05, lp2=0.95):

    # Calculate the Pearson correlation coefficient
    corr_coeff, p_value = pearsonr(x, y)
    print("Correlation coefficient:", corr_coeff)
    print("p-value:", p_value)
    
    # Fit a linear regression model
    x1 = np.array(x).reshape(-1, 1)  # Reshape for scikit-learn compatibility
    model = LinearRegression().fit(x1, y)

    # Create a scatter plot of the data
    plt.figure(figsize=(10, 5))
    plt.scatter(x, y, alpha=0.7)

    # Add a regression line
    plt.plot(np.unique(x), model.predict(np.unique(x).reshape(-1, 1)), color='black')

    # Add axis labels
    plt.xlabel(x_label, fontsize=20)
    plt.ylabel(y_label, fontsize=20)
    plt.xticks(fontsize=15)
    plt.yticks(fontsize=15)

    # Add correlation coefficient and p-value inside the plot

    plt.text(lp1, lp2, f"r: {corr_coeff:.2f}, p: {'< .01' if p_value < 0.01 else f'{p_value:.3f}'.lstrip('0')}", 
             fontsize=15, ha='left', va='top', fontweight='bold', style='italic', transform=plt.gca().transAxes)


    plt.show()

    return {
        "correlation_coefficient": corr_coeff,
        "p_value": p_value,
        "regression_model": model
    }

## ATAS Metrics vs SSI and %SS scores (Figure 4)

In [ ]:
# ---------- helpers for plot-only formatting ----------

def _fmt_no_leading_zero(x, ndigits, use_apa_minus=True):
    """
    APA-style numeric formatting for plot text:
    0.48   -> .48
    -0.48  -> −.48   (or -.48 if use_apa_minus=False)
    0.012  -> .012
    -0.012 -> −.012
    """
    if pd.isna(x):
        return "nan"

    s = f"{x:.{ndigits}f}"

    # remove leading zero for positive decimals
    if s.startswith("0."):
        s = s[1:]  # "0.48" -> ".48"

    # remove leading zero for negative decimals
    elif s.startswith("-0."):
        s = "-" + s[2:]  # "-0.48" -> "-.48"
        if use_apa_minus:
            s = s.replace("-", "−", 1)  # APA minus sign

    return s



# ---------- wrapped plotting functions (PLOTS ONLY) ----------
def analyze_and_plot_correlation_fmt(x, y, x_label="X-axis", y_label="Y-axis", lp1=0.05, lp2=0.95):
    corr_coeff, p_value = pearsonr(x, y)

    x1 = np.array(x).reshape(-1, 1)
    model = LinearRegression().fit(x1, y)

    plt.figure(figsize=(10, 5))
    plt.scatter(x, y, alpha=0.7)
    plt.plot(np.unique(x), model.predict(np.unique(x).reshape(-1, 1)), color='black')

    plt.xlabel(x_label, fontsize=20)
    plt.ylabel(y_label, fontsize=20)
    plt.xticks(fontsize=15)
    plt.yticks(fontsize=15)

    r_txt = _fmt_no_leading_zero(corr_coeff, 2)
    p_txt = "< .01" if p_value < 0.01 else _fmt_no_leading_zero(p_value, 2)

    plt.text(
        lp1, lp2,
        f"r: {r_txt}, p: {p_txt}",
        fontsize=15, ha='left', va='top',
        fontweight='bold', style='italic',
        transform=plt.gca().transAxes
    )

    plt.show()


def analyze_and_plot_correlation_unq_fmt(x, y, x_label="X-axis", y_label="Y-axis", lp1=0.05, lp2=0.95):
    corr_coeff, p_value = pearsonr(x, y)

    x1 = np.array(x).reshape(-1, 1)
    model = LinearRegression().fit(x1, y)

    plt.figure(figsize=(10, 5))
    plt.scatter(x, y, alpha=0.7)
    plt.plot(np.unique(x), model.predict(np.unique(x).reshape(-1, 1)), color='black')

    plt.xlabel(x_label, fontsize=20)
    plt.ylabel(y_label, fontsize=20)
    plt.xticks(fontsize=15)
    plt.yticks(fontsize=15)

    r_txt = _fmt_no_leading_zero(corr_coeff, 2)
    p_txt = "< .01" if p_value < 0.01 else _fmt_no_leading_zero(p_value, 3)

    plt.text(
        lp1, lp2,
        f"r: {r_txt}, p: {p_txt}",
        fontsize=15, ha='left', va='top',
        fontweight='bold', style='italic',
        transform=plt.gca().transAxes
    )

    plt.show()


# ---------- helper: compute r/p, plot only if significant ----------
def plot_correlation_if_significant(x, y, plot_func, x_label, y_label, name,
                                   alpha=0.05, lp1=0.05, lp2=0.95):
    xy = pd.DataFrame({'x': x, 'y': y}).dropna()
    if xy.shape[0] < 3:
        return {'name': name, 'n': xy.shape[0], 'r': np.nan, 'p': np.nan, 'plotted': False}

    r, p = pearsonr(xy['x'], xy['y'])
    plotted = False
    if p < alpha:
        plot_func(xy['x'], xy['y'], x_label=x_label, y_label=y_label, lp1=lp1, lp2=lp2)
        plotted = True

    return {'name': name, 'n': xy.shape[0], 'r': r, 'p': p, 'plotted': plotted}


def run_correlations_significant_only(alpha=0.05):
    results = []

    cws_SSI = par_details[par_details['Group'] == 'CWS']['SSI']
    cws_SS_per = par_details[par_details['Group'] == 'CWS']['%SS']

    metric_specs = [
        dict(col='Speech_Duration_s',        xlab="Total Vocal Duration (s)",             lp1=0.58, lp2=0.95),
        dict(col='Pause_Duration_s',         xlab="Total Pause Duration (s)",             lp1=0.58, lp2=0.95),
        dict(col='Mean Speech_s',            xlab="Mean Speech Duration (s)",             lp1=0.58, lp2=0.95),
        dict(col='CV Speech',                xlab="Speech Duration Variability",          lp1=0.58, lp2=0.95),
        dict(col='CV Pause',                 xlab="Pause Duration Variability",           lp1=0.58, lp2=0.95),
        dict(col='long_p_durations_mean',    xlab="Long Pause Duration Mean (s)",         lp1=0.58, lp2=0.95),
        dict(col='short_p_durations_mean',   xlab="Short Pause Duration Mean (s)",        lp1=0.58, lp2=0.95),
        dict(col='long_p_durations_cv',      xlab="Long Pause Duration Variability",      lp1=0.58, lp2=0.95),
        dict(col='short_p_durations_cv',     xlab="Short Pause Duration Variability",     lp1=0.58, lp2=0.95),
        dict(col='Speech_Rate',              xlab="Speech Rate",                          lp1=0.58, lp2=0.95),
        dict(col='Pause_Events',             xlab="Pause Count",                          lp1=0.05, lp2=0.95),
        dict(col='long_p_count',             xlab="Long Pause Count",                     lp1=0.05, lp2=0.95),
        dict(col='short_p_count',            xlab="Short Pause Count",                    lp1=0.05, lp2=0.95),
        dict(col='Mean Pause_s',             xlab="Mean Pause Duration (s)",              lp1=0.58, lp2=0.95),
    ]

    def pick_plot_func(col, y_label):
        if col == 'Speech_Rate' and y_label == '%SS':
            return analyze_and_plot_correlation_unq_fmt
        return analyze_and_plot_correlation_fmt

    for spec in metric_specs:
        col = spec['col']
        x = par_details[par_details['Group'] == 'CWS'][col]

        results.append(plot_correlation_if_significant(
            x, cws_SS_per,
            pick_plot_func(col, '%SS'),
            x_label=spec['xlab'], y_label="%SS",
            name=f"CWS: {col} vs %SS",
            alpha=alpha, lp1=spec['lp1'], lp2=spec['lp2']
        ))

        results.append(plot_correlation_if_significant(
            x, cws_SSI,
            pick_plot_func(col, 'SSI'),
            x_label=spec['xlab'], y_label="SSI",
            name=f"CWS: {col} vs SSI",
            alpha=alpha, lp1=spec['lp1'], lp2=spec['lp2']
        ))

    corr_table = pd.DataFrame(results)
    corr_table['metric'] = corr_table['name'].str.extract(r"CWS:\s*(.*?)\s*vs")[0]
    corr_table['outcome'] = corr_table['name'].str.extract(r"vs\s*(%SS|SSI)")[0]

    corr_table = corr_table[['metric', 'outcome', 'n', 'r', 'p', 'plotted', 'name']] \
        .sort_values(['outcome', 'p'], na_position='last') \
        .reset_index(drop=True)

    print("Correlation results (all tested):")
    display(corr_table)

    print(f"Significant correlations only (p < {alpha}):")
    display(corr_table[corr_table['p'] < alpha].reset_index(drop=True))

    return corr_table


# --- Run ---
corr_summary = run_correlations_significant_only(alpha=0.05)


## ATAS Metrics vs Age (Figure 3)

In [ ]:
# ---------- helpers ----------
def _fmt_no_leading_zero(x, ndigits, use_apa_minus=True):
    if pd.isna(x):
        return "nan"

    s = f"{x:.{ndigits}f}"

    if s.startswith("0."):
        s = s[1:]
    elif s.startswith("-0."):
        s = "-" + s[2:]
        if use_apa_minus:
            s = s.replace("-", "−", 1)

    return s


def _fmt_p_value(p):
    if pd.isna(p):
        return "nan"
    return "< .01" if p < 0.01 else _fmt_no_leading_zero(p, 2)


def _get_group_mask(series, group_name):
    s = series.astype(str).str.strip().str.upper()
    return s == group_name.upper()


def _normalize_xy(x, y):
    x = np.asarray(x, dtype=float)
    y = np.asarray(y, dtype=float)

    if len(x) == 0:
        return np.array([]), np.array([])

    x_norm = (x - x.min()) / (x.max() - x.min() + 1e-9)
    y_norm = (y - y.min()) / (y.max() - y.min() + 1e-9)
    return x_norm, y_norm


def _choose_legend_location(x, y):
    x_norm, y_norm = _normalize_xy(x, y)

    if len(x_norm) == 0:
        return "upper right"

    top_right = np.sum((x_norm > 0.6) & (y_norm > 0.6))
    bottom_right = np.sum((x_norm > 0.6) & (y_norm < 0.4))
    bottom_left = np.sum((x_norm < 0.4) & (y_norm < 0.4))

    counts = {
        "upper right": top_right,
        "lower right": bottom_right,
        "lower left": bottom_left
    }

    return min(counts, key=counts.get)


def _legend_box_axes_coords(loc):
    """
    Approximate legend box region in axes coordinates.
    Used only to keep the annotation away from the legend.
    """
    boxes = {
        "upper right": (0.74, 0.78, 0.24, 0.18),
        "lower right": (0.74, 0.04, 0.24, 0.18),
        "lower left":  (0.02, 0.04, 0.24, 0.18),
        "upper left":  (0.02, 0.78, 0.24, 0.18),
    }
    return boxes.get(loc, (0.74, 0.78, 0.24, 0.18))


def _count_points_in_rect(x_norm, y_norm, x0, y0, w, h):
    return np.sum(
        (x_norm >= x0) & (x_norm <= x0 + w) &
        (y_norm >= y0) & (y_norm <= y0 + h)
    )


def _rectangles_overlap(r1, r2):
    x1, y1, w1, h1 = r1
    x2, y2, w2, h2 = r2
    return not (
        x1 + w1 < x2 or
        x2 + w2 < x1 or
        y1 + h1 < y2 or
        y2 + h2 < y1
    )


def _choose_annotation_position(x, y, legend_loc, box_w=0.34, box_h=0.16):
    """
    Search a dense grid of candidate text boxes and choose the emptiest region.
    Returns:
      text_x, text_y, ha, va
    where text_x/text_y are the anchor coords for plt.text in axes units.
    """
    x_norm, y_norm = _normalize_xy(x, y)

    if len(x_norm) == 0:
        return 0.05, 0.95, "left", "top"

    legend_rect = _legend_box_axes_coords(legend_loc)

    # Candidate lower-left positions for the annotation box
    xs = np.linspace(0.03, 0.97 - box_w, 22)
    ys = np.linspace(0.03, 0.97 - box_h, 16)

    best_score = None
    best_rect = None

    for x0 in xs:
        for y0 in ys:
            rect = (x0, y0, box_w, box_h)

            # skip overlapping with legend box
            if _rectangles_overlap(rect, legend_rect):
                continue

            # base score = how many points are inside the proposed text box
            point_count = _count_points_in_rect(x_norm, y_norm, x0, y0, box_w, box_h)

            # mild penalty for being very close to dense center
            cx = x0 + box_w / 2
            cy = y0 + box_h / 2
            center_penalty = np.sum(
                ((x_norm - cx) / (box_w / 1.5))**2 +
                ((y_norm - cy) / (box_h / 1.5))**2 < 1
            )

            # slight preference for top placements if equally empty
            top_bonus = -0.15 * cy

            score = point_count + 0.35 * center_penalty + top_bonus

            if best_score is None or score < best_score:
                best_score = score
                best_rect = rect

    if best_rect is None:
        return 0.05, 0.95, "left", "top"

    x0, y0, w, h = best_rect

    # Use lower-left anchor with va='bottom' for searched rectangle
    return x0, y0, "left", "bottom"


# ---------- plotting ----------
def analyze_and_plot_correlation_age_groupwise(
    df,
    x_col,
    y_col,
    group_col="Group",
    x_label="Age (months)",
    y_label="Y-axis",
    cws_color="cornflowerblue",
    cwns_color="yellowgreen"
):

    plot_df = df[[x_col, y_col, group_col]].dropna()
    ms_metrics = ["Mean Speech_s", "short_p_durations_mean", "Mean Pause_s", "long_p_durations_mean"]

    if y_col in ms_metrics:
        plot_df[y_col] = plot_df[y_col] * 1000

    cws_df = plot_df[_get_group_mask(plot_df[group_col], "CWS")]
    cwns_df = plot_df[_get_group_mask(plot_df[group_col], "CWNS")]

    def _corr(subdf):
        if len(subdf) < 3:
            return np.nan, np.nan
        return pearsonr(subdf[x_col], subdf[y_col])

    cws_r, cws_p = _corr(cws_df)
    cwns_r, cwns_p = _corr(cwns_df)

    # ---------- FIXED FIGURE SIZE ----------
    plt.figure(figsize=(11, 5))

    # ---------- FIXED AXES BOX ----------
    ax = plt.gca()
    ax.set_position([0.1, 0.15, 0.8, 0.75])

    # scatter
    plt.scatter(
        cwns_df[x_col], cwns_df[y_col],
        color=cwns_color, alpha=0.75, label="CWNS"
    )

    plt.scatter(
        cws_df[x_col], cws_df[y_col],
        color=cws_color, alpha=0.75, label="CWS", marker="^",
        s=50   
    )

    # regression lines
    def _plot_line(subdf, color):
        if len(subdf) < 2:
            return
        x = subdf[x_col].values.reshape(-1, 1)
        y = subdf[y_col].values
        model = LinearRegression().fit(x, y)

        x_line = np.linspace(subdf[x_col].min(), subdf[x_col].max(), 100)
        y_line = model.predict(x_line.reshape(-1, 1))
        plt.plot(x_line, y_line, color=color, linewidth=2)

    _plot_line(cwns_df, cwns_color)
    _plot_line(cws_df, cws_color)

    # labels
    plt.xlabel(x_label, fontsize=20)
    plt.ylabel(y_label, fontsize=20)
    plt.xticks(fontsize=15)
    plt.yticks(fontsize=15)

    combined_x = pd.concat([cws_df[x_col], cwns_df[x_col]], ignore_index=True)
    combined_y = pd.concat([cws_df[y_col], cwns_df[y_col]], ignore_index=True)

    # ---------- DYNAMIC LEGEND ----------
    legend_loc = _choose_legend_location(combined_x, combined_y)

    plt.legend(
        fontsize=12,
        loc=legend_loc,
        frameon=True,
        fancybox=False,
        framealpha=1,
        edgecolor='lightgray',
        facecolor='#f2f2f2',
        borderpad=0.1,
        labelspacing=0.2,
        handletextpad=0.1,
        borderaxespad=0.1
    )

    # ---------- EMPTIEST-SPACE ANNOTATION ----------
    ann_x, ann_y, ann_ha, ann_va = _choose_annotation_position(
        combined_x, combined_y, legend_loc, box_w=0.34, box_h=0.16
    )

    annotation = (
        f"CWNS: r = {_fmt_no_leading_zero(cwns_r, 2)}, p = {_fmt_p_value(cwns_p)}\n"
        f"CWS:  r = {_fmt_no_leading_zero(cws_r, 2)}, p = {_fmt_p_value(cws_p)}"
    )

    plt.text(
        ann_x, ann_y,
        annotation,
        color='black',
        fontsize=14,
        fontweight='bold',
        ha=ann_ha,
        va=ann_va,
        transform=ax.transAxes,
        bbox=dict(
            facecolor='white',
            edgecolor='none',
            alpha=0.75,
            pad=0.2
        )
    )

    plt.margins(x=0.05)
    plt.show()

    return {
        "CWNS_r": cwns_r,
        "CWNS_p": cwns_p,
        "CWS_r": cws_r,
        "CWS_p": cws_p
    }


# ---------- runner ----------
def run_age_correlations_groupwise():

    results = []

    metric_specs_age = [
        dict(col='Speech_Rate', ylab="Speech Rate"),
        dict(col='Speech_Duration_s', ylab="Total Vocal Duration (s)"),
        dict(col='Pause_Duration_s', ylab="Total Pause Time (s)"),
        dict(col='Mean Speech_s', ylab="Mean Vocal Duration (ms)"),
        dict(col='Mean Pause_s', ylab="Mean Pause Duration (ms)"),
        dict(col='CV Speech', ylab="Vocal Duration Variability"),
        dict(col='CV Pause', ylab="Pause Duration Variability"),
        dict(col='long_p_durations_mean', ylab="Mean Long Pause Duration (ms)"),
        dict(col='long_p_durations_cv', ylab="Long Pause Duration Variability"),
        dict(col='short_p_durations_mean', ylab="Mean Short Pause Duration (ms)"),
        dict(col='short_p_durations_cv', ylab="Short Pause Duration Variability"),
        dict(col='Pause_Events', ylab="No. of Pauses"),
        dict(col='long_p_count', ylab="No. of Long Pauses"),
        dict(col='short_p_count', ylab="No. Short Pauses"),
    ]

    for spec in metric_specs_age:
        res = analyze_and_plot_correlation_age_groupwise(
            df=par_details,
            x_col="Age",
            y_col=spec['col'],
            y_label=spec['ylab']
        )
        res['metric'] = spec['col']
        results.append(res)

    return pd.DataFrame(results)


# ---------- RUN ----------
age_corr_summary_groupwise = run_age_correlations_groupwise()

## Check correlations (metrics significantly different between groups)

In [ ]:
def _fmt_no_leading_zero(x, ndigits=2, use_apa_minus=True):
    if pd.isna(x):
        return "nan"

    s = f"{x:.{ndigits}f}"

    if s.startswith("0."):
        s = s[1:]
    elif s.startswith("-0."):
        s = "-" + s[2:]
        if use_apa_minus:
            s = s.replace("-", "−", 1)

    return s


def _fmt_p_value(p):
    if pd.isna(p):
        return "nan"
    return "< .01" if p < 0.01 else _fmt_no_leading_zero(p, 2)


def pairwise_metric_correlations(df):
    metric_cols = {
        "Speech rate": "Speech_Rate",
        "Total pause duration": "Pause_Duration_s",
        "Mean vocal duration": "Mean Speech_s",
        "Total pauses": "Pause_Events",
        "Long pauses": "long_p_count",
        "Short pauses": "short_p_count",
    }

    results = []

    for m1, m2 in combinations(metric_cols.keys(), 2):
        col1 = metric_cols[m1]
        col2 = metric_cols[m2]

        tmp = df[[col1, col2]].dropna()
        n = len(tmp)

        if n >= 3:
            r, p = pearsonr(tmp[col1], tmp[col2])
        else:
            r, p = np.nan, np.nan

        results.append({
            "Metric 1": m1,
            "Metric 2": m2,
            "N": n,
            "r": r,
            "p": p,
            "r (APA)": _fmt_no_leading_zero(r, 2) if not pd.isna(r) else "nan",
            "p (APA)": _fmt_p_value(p)
        })

    results_df = pd.DataFrame(results).sort_values("p", na_position="last").reset_index(drop=True)

    print("Pairwise correlations among the six ATAS metrics:")
    display(results_df)

    return results_df


# ---- run ----
pairwise_corr_6 = pairwise_metric_correlations(par_details)

## Combined Plots (Figure 5)

In [ ]:
long_p_count_cwns = par_details[par_details['Group'] == 'CWNS']['long_p_count']
short_p_count_cwns = par_details[par_details['Group'] == 'CWNS']['short_p_count']
long_p_count_cws = par_details[par_details['Group'] == 'CWS']['long_p_count']
short_p_count_cws = par_details[par_details['Group'] == 'CWS']['short_p_count']

long_p_count_cws = np.array(long_p_count_cws)
short_p_count_cws = np.array(short_p_count_cws)
long_p_count_cwns = np.array(long_p_count_cwns)
short_p_count_cwns = np.array(short_p_count_cwns)


cws_SSI = par_details[par_details['Group'] == 'CWS']['SSI']
cws_SS_per = par_details[par_details['Group'] == 'CWS']['%SS']
# Labels (CWS only) — assumed already defined earlier too
cws_SSI    = np.array(cws_SSI)
cws_SS_per = np.array(cws_SS_per)


speech_rate_cwns =  par_details[par_details['Group'] == 'CWNS']['Speech_Rate']
speech_rate_cws =  par_details[par_details['Group'] == 'CWS']['Speech_Rate']
pause_events_cwns = par_details[par_details['Group'] == 'CWNS']['Pause_Events']
pause_events_cws = par_details[par_details['Group'] == 'CWS']['Pause_Events']

speech_rate_cwns  = np.array(speech_rate_cwns)
pause_events_cwns = np.array(pause_events_cwns)
speech_rate_cws   = np.array(speech_rate_cws)
pause_events_cws  = np.array(pause_events_cws)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt


# -----------------------------
# Helper to plot + smart labels
# -----------------------------
def scatter_with_cws_labels(x_cwns, y_cwns, x_cws, y_cws, labels,
                            title, xlabel, ylabel,
                            radius_px=6, dy_points=10):

    fig, ax = plt.subplots(figsize=(10, 10))

    # CWNS (circles)
    ax.scatter(x_cwns, y_cwns, color="yellowgreen", s=100, label="CWNS")

    # CWS (stars)
    ax.scatter(x_cws, y_cws, color="cornflowerblue", s=100, marker="^", label="CWS")

    labels = np.round(np.asarray(labels), 2)

    # --- draw once so transforms are valid ---
    fig.canvas.draw()

    # Data → display (pixels) for CWS points only
    pts  = np.column_stack([x_cws, y_cws])
    disp = ax.transData.transform(pts)

    # ---- group by proximity in pixel space (handles exact duplicates & near overlaps) ----
    groups = []
    for i, p in enumerate(disp):
        placed = False
        for g in groups:
            if np.hypot(*(p - disp[g[0]])) <= radius_px:
                g.append(i)
                placed = True
                break
        if not placed:
            groups.append([i])

    arrow = dict(
        arrowstyle="->", lw=1, color="gray",
        shrinkA=0, shrinkB=2, connectionstyle="arc3,rad=0"
    )

    ax_bbox = ax.get_window_extent()
    pt_to_px = fig.dpi / 72.0

    for g in groups:
        g = sorted(g, key=lambda idx: pts[idx, 1])  # optional neat stacking
        for j, idx in enumerate(g):
            sign = +1 if j % 2 == 0 else -1
            mag  = 1 + (j // 2)
            dy   = sign * mag * dy_points

            x, y = pts[idx]
            y_pix = disp[idx, 1]
            label_y_pix = y_pix + dy * pt_to_px

            # keep label inside axes: flip if it would cross the top/bottom
            if label_y_pix > ax_bbox.y1:
                dy = -abs(dy)
            elif label_y_pix < ax_bbox.y0:
                dy = abs(dy)

            va = "bottom" if dy > 0 else "top"

            ax.annotate(
                str(labels[idx]),
                xy=(x, y),
                xytext=(0, dy),
                textcoords="offset points",
                ha="center", va=va, fontsize=12,
                arrowprops=arrow,
                clip_on=True, annotation_clip=True
            )

    ax.legend(fontsize=20)
    ax.set_title(title, fontsize=20)
    ax.set_xlabel(xlabel, fontsize=20)
    ax.set_ylabel(ylabel, fontsize=20)
    ax.tick_params(labelsize=15)

    fig.tight_layout()
    plt.show()


# -----------------------------
# Plot 1: Speech_Rate vs Pause_Events with %SS labels (CWS only)
# -----------------------------
scatter_with_cws_labels(
    x_cwns=speech_rate_cwns,  y_cwns=pause_events_cwns,
    x_cws=speech_rate_cws,    y_cws=pause_events_cws,
    labels=cws_SS_per,
    title="Speech Rate and Pause Events with %SS",
    xlabel="Speech Rate",
    ylabel="Pause Count"
)

# -----------------------------
# Plot 2: Speech_Rate vs Pause_Events with SSI labels (CWS only)
# -----------------------------
scatter_with_cws_labels(
    x_cwns=speech_rate_cwns,  y_cwns=pause_events_cwns,
    x_cws=speech_rate_cws,    y_cws=pause_events_cws,
    labels=cws_SSI,
    title="Speech Rate and Pause Events with SSI",
    xlabel="Speech Rate",
    ylabel="Pause Count"
)

In [ ]:

# -----------------------------
# Helper to plot + smart labels
# -----------------------------
def scatter_with_cws_labels(x_cwns, y_cwns, x_cws, y_cws, labels,
                            title, xlabel, ylabel,
                            radius_px=6, dy_points=10):

    fig, ax = plt.subplots(figsize=(10, 10))

    # CWNS (circles)
    ax.scatter(x_cwns, y_cwns, color="yellowgreen", s=100, label="CWNS")

    # CWS (stars)
    ax.scatter(x_cws, y_cws, color="cornflowerblue", s=100, marker="^", label="CWS")

    labels = np.round(np.asarray(labels), 2)

    # --- draw once so transforms are valid ---
    fig.canvas.draw()

    # Data → display (pixels) for CWS points only (since we label only those)
    pts  = np.column_stack([x_cws, y_cws])
    disp = ax.transData.transform(pts)

    # ---- group by proximity in pixel space (handles exact duplicates & near overlaps) ----
    groups = []
    for i, p in enumerate(disp):
        placed = False
        for g in groups:
            if np.hypot(*(p - disp[g[0]])) <= radius_px:
                g.append(i)
                placed = True
                break
        if not placed:
            groups.append([i])

    arrow = dict(
        arrowstyle="->", lw=1, color="gray",
        shrinkA=0, shrinkB=2, connectionstyle="arc3,rad=0"
    )

    ax_bbox = ax.get_window_extent()
    pt_to_px = fig.dpi / 72.0

    for g in groups:
        g = sorted(g, key=lambda idx: pts[idx, 1])  # optional neat stacking
        for j, idx in enumerate(g):
            sign = +1 if j % 2 == 0 else -1
            mag  = 1 + (j // 2)
            dy   = sign * mag * dy_points

            x, y = pts[idx]
            y_pix = disp[idx, 1]
            label_y_pix = y_pix + dy * pt_to_px

            # keep label inside axes: flip if it would cross the top/bottom
            if label_y_pix > ax_bbox.y1:
                dy = -abs(dy)
            elif label_y_pix < ax_bbox.y0:
                dy = abs(dy)

            va = "bottom" if dy > 0 else "top"

            ax.annotate(
                str(labels[idx]),
                xy=(x, y),
                xytext=(0, dy),
                textcoords="offset points",
                ha="center", va=va, fontsize=12,
                arrowprops=arrow,
                clip_on=True, annotation_clip=True
            )

    ax.legend(fontsize=20)
    ax.set_title(title, fontsize=20)
    ax.set_xlabel(xlabel, fontsize=20)
    ax.set_ylabel(ylabel, fontsize=20)
    ax.tick_params(labelsize=15)

    fig.tight_layout()
    plt.show()


# -----------------------------
# Plot 1: Short vs Long Pauses with %SS labels (CWS only)
# -----------------------------
scatter_with_cws_labels(
    x_cwns=short_p_count_cwns, y_cwns=long_p_count_cwns,
    x_cws=short_p_count_cws,   y_cws=long_p_count_cws,
    labels=cws_SS_per,
    title="Long and Short Pauses with %SS",
    xlabel="Short Pause Count",
    ylabel="Long Pause Count"
)

# -----------------------------
# Plot 2: Short vs Long Pauses with SSI labels (CWS only)
# -----------------------------
scatter_with_cws_labels(
    x_cwns=short_p_count_cwns, y_cwns=long_p_count_cwns,
    x_cws=short_p_count_cws,   y_cws=long_p_count_cws,
    labels=cws_SSI,
    title="Long and Short Pauses with SSI",
    xlabel="Short Pause Count",
    ylabel="Long Pause Count"
)